In diesem Notebook habe ich das BERT-Modell darauf finegetuned Unterbrechungen in Reden zu erkennen. Die Klassifikation findet auf Satzebene statt (und nicht auf Tokenebene, wie bei [My_BERT_Model.ipynb](./My_BERT_Model.ipynb) ) Der genaue Prozess wird folgend beschrieben.

In [14]:
!pip install datasets
!pip install torch
!pip install evaluate
!pip install transformers==4.28.1
!pip install accelerate==0.15.0
!pip install tokenizers==0.13.3

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Defaulting to user installation because normal site-packages is not writeable


Die folgende Zelle segmentiert die Daten anhand des "interruption" Tokens in Segmente. Es werden sogenannte Kontext- und Einwurfsegmente erstellt. Kontextsegmente sind die Daten, die vor dem Kommentar stehen. Einwurfsegmente sind die, die nach dem Einwurf und vor dem nächsten Punkt stehen.

Eine kleine Besonderheit ist der Umgang mit dem letzten Segment der Daten. Das letzte Segment muss nochmal extra behandelt werden, da die Segmente, die sich direkt vor und nach einem "interruption" Token befinden, logisch verknüpft sind. Dieses Segment steht hinter dem letzten "interruption" Token und wird nicht durch einen weiteren Token gefolgt. Es muss als Kontextsegment behandelt werden, da es keine weitere Unterbrechung geben wird.

Anschließend wird jedes Segment entsprechend seiner Position mit Labels versehen. Kontextsegmente bekommen das Label 0 und Einwurfsegmente das Label 1.

Zum Schluss werden aus diesen Daten Listen von Segmenten und den zugehörigen Labels erstellt. Diese werden dann im Anschluss geflattet, damit die Daten direkt in ein PandasDataframe umzuwandeln. In dieser Form sind die Daten leichter handhabbar und optimal für die Weiterverarbeitung und das Training geeignet.

Im Output kann man dann die Gesamtzahl der Datensätze im Dateframe sehen.

Anmerkung: Ich habe mehrmals getestet, welche Anzahl von Daten am effizientesten ist. Bei 1000 Einträgen ist das Verhältnis des Zeitaufwandes vom Training zur Qualität des Outputs für mich am besten gewesen.

In [3]:
import json
import re
import pandas as pd
from transformers import AutoTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

file_path = '/var/lib/private/simonjaned87761/team-04_deprecated/data/opendata_api/modified_texts_with_interruption.json'

with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

original_data = data.copy()
mod_data = []

for entry in original_data[:1000]: 
    modified_text = entry['modified_text']
    modified_text = re.sub(r'\s+', ' ', modified_text)  # Entfernt überflüssige Leerzeichen
    segments = modified_text.split('<interruption>')
    
    texts = [] 
    labels = [] 

    for i in range(len(segments) - 1):  # Alle Segmente außer das letzte
        context_segment = segments[i].strip()  # Segment vor dem Einwurf
        interruption_segment = segments[i + 1].strip().split('.')[0]  # Einwurf bis zum nächsten Punkt
        
        if context_segment:
            texts.append(context_segment)
            labels.append(0)  # Kontextsegment

        if interruption_segment:
            texts.append(interruption_segment)
            labels.append(1)  # Einwurfsegment

    # Letztes Segment hinzufügen, wenn es nicht leer ist
    last_segment = segments[-1].strip()
    if last_segment:
        texts.append(last_segment)
        labels.append(0)  # Letztes Segment ist kein Einwurf

    # Speichern der modifizierten Einträge
    mod_data.append({
        'segments': texts,
        'labels': labels
    })

texts = [seg for entry in mod_data for seg in entry['segments']]
labels = [label for entry in mod_data for label in entry['labels']]

df = pd.DataFrame({'text': texts, 'label': labels})

print(f"Complete dataset: {len(df)}")

Complete dataset: 5364


Anschließend werden die Daten des PandasDataframes in 80% Trainings- und 20% Testdaten unterteilt. Setzt man den Parameter random_state nicht auf 42, so wird beim Ausführen des Codes jedes mal ein anderes Testset generiert. Im Laufe der Zeit wird das Modell in der Lage sein den gesamten Datensatz zu erkennen, was wir vermeiden wollen.
Die PandasDataframes müssen in Huggung Face Datasets konvertiert werden, damit man z.B. den Trainer ohne Probleme verwenden kann und generell die Kompatibilität gewährleistet. Ähnlich ist es mit dem DatasetDict. Die Trainer API erwartet eine DatasetDict-Eingabe, weshalb die Trainings- und Testdaten als Einträge vorhanden sind.

In [5]:
train_df = df.sample(frac=0.80, random_state=42)  # frac = 0.80 = 80% Trainingsdaten
test_df = df.drop(train_df.index)  #20% Testdaten

# Konvertiere die DataFrames zu Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# Das ist nur zum Debuggen, bzw. Überblick behalten über die Daten
print(type(dataset_dict))
print(dataset_dict["train"][10])

<class 'datasets.dataset_dict.DatasetDict'>
{'text': 'der einerseits im Bundesrat die Erhöhung der Einkommensgrenzen im sozialen Wohnungsbau einfordert und sich jetzt andererseits darüber beklagt, daß diese Grenzen im Rahmen unseres Gesetzentwurfs angehoben werden', 'label': 1}


Hier initialisiere ich BERT-base-german-case. 

Die Zeile: "model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)" ist für dieses Notebook äußerst ausschlaggebend, da hier das vortrainierte BERT-Modell auf die gewünschte binäre Klassifikationsaufgabe angepasst wird. 

Innerhalb der Variable "tokenized_datasets" wird die tokenize_function() durch map() auf jeden der 1000 Einträge angewendet. Dadurch werden alle Daten tokenisiert. Ich habe mich dafür entschieden die Verarbeitung in Batches durchzuführen, um Zeit zu sparen.

In [6]:
from transformers import pipeline

# mein BERT-Modell und den zugehörigen Tokenizer laden
model_name = 'dbmdz/bert-base-german-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

/var/lib/simonjaned87761/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at dbmdz/bert-base-german-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the

Map:   0%|          | 0/4291 [00:00<?, ? examples/s]

Map:   0%|          | 0/1073 [00:00<?, ? examples/s]

Folgend wird das Modell trainiert. Hierfür wurde eine seed value von 42 verwendet. Diese ist vergleichbar mit dem random state. Diese seed value sorgt auch für die Reproduziertbarkeit von Zufallszahlen. In dieser Zelle z.B. beim Initialisieren der Gewichte.
Die Trainingsargumente sind der nächste Teil. Dort habe ich festgelegt, dass das Modell nach jeder Epoche evaluiert wird. Die Initialisierung der Batch Size war mir zusätzlich wichtig, damit das Training effizienter verläuft. 
Die Funktion compute_metrics() berechnet die Accuracy, indem es die vorhergesagten und tatsächlichen Labels lädt und darauf die Funktion metric.compute() anwendet.
Anschließend wird der Trainer initialisiert.

In [8]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

seed_value = 42

training_args = TrainingArguments(
    output_dir="./results", 
    evaluation_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    seed=seed_value,  # seed_value wird hier gesetzt
    logging_steps=10,
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.059000,0.049815,0.983225
2,0.018700,0.052948,0.979497


TrainOutput(global_step=1074, training_loss=0.07327754536402556, metrics={'train_runtime': 2288.0286, 'train_samples_per_second': 3.751, 'train_steps_per_second': 0.469, 'total_flos': 564504769274880.0, 'train_loss': 0.07327754536402556, 'epoch': 2.0})

Die Evaluation ist sehr gut ausgefallen, was auch später an den Ergebnissen erkennbar ist. Der Loss ist mit ca. 0,05 sehr gut und die Accuracy mit ca. 0,98 erstaunlich präzise. 

In [7]:
# Evaluierung des Modells
eval_result = trainer.evaluate()
print(f"Eval results: {eval_result}")

Eval results: {'eval_loss': 0.04967799410223961, 'eval_accuracy': 0.983224603914259, 'eval_runtime': 62.7173, 'eval_samples_per_second': 17.109, 'eval_steps_per_second': 2.153, 'epoch': 1.0}


Aufgrund der langen Ladezeiten habe ich mich der Zwischenspeicherung des Modells beholfen. In diesem Codeblock speicher ich das fine-tuned Modell, sowie den Tokenizer auf der Festplatte, damit ich sie danach wieder laden kann.

In [8]:
model_path = "./models/finetuned_bert_german_cased"
tokenizer_path = model_path

trainer.save_model(model_path)
tokenizer.save_pretrained(tokenizer_path)

model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

Hier wird das Modell getestet. Ich habe ähnlich strukturierten Beispieltext, wie die Trainingsdaten als Input für das Modell verwendet. Ich habe dafür mehrere Sätze als Liste abgespeichert, die das Modell satzweise als Einwurf oder keinen Einwurf klassifizieren konnte.
Bei der Tokenisierung ist zu bemerken, dass ich Padding eingestellt hab. D.h. alle Texte werden auf die gleiche Länge der längsten Sequenz aufgefüllt. Kürzere Sequenzen werden mit dem Padding-Token [PAD] aufgefüllt. 

Im Output sieht man zuerst die Indizes der Sätze mit den vorhergesagten Einwürfen. Darunter sieht man Sätze hinter denen die Vorhersage steht. Die Anzeige der Indizes soll Zeit sparen. Man kann bspw. bei Index 3 einfach auf den 4. Satz gucken und sehen, ob er sinnvoll als Einwurf klassifiziert wurde. Was noch einfacher ist, ist mit Strg + F nach "Einwurf" suchen und dann die markierten Vorhersagen anschauen (weil hier der Inputtext etwas lang ist). Anfangs sieht man, dass einige Einwürfe falsch klassifiziert wurden. Aber weiter unten wurden Beifall und Zurufe weitestgehend richtig erkannt. Die Sätze, die keinen Kommentar enthalten, wurden nach meinem Augenmaß immer richtig als "Kein Einwurf" klassifiziert. Wie genau mein Blick war, wird sich in der nächsten Zelle bei der Erläuterung der Precision und Recall klären.

Anzumerken ist, dass torch.no_grad() die Berechnung der Gradienten deaktiviert, um Speicher zu sparen und die Berechnung zu beschleunigen, da keine Backpropagation durchgeführt wird. 
model(**inputs) führt einen Forward Pass durch das Modell mit den tokenisierten Eingabetexten durch.
logits.numpy() konvertiert die PyTorch-Tensoren in NumPy-Arrays und darauf den argmax angewendet resultiert dann mit dem höchsten Logit-Wert der letzten Achse. Dies stellt dann die vorhergesagte Klasse dar. (Logits sind einfach die nicht normalisierten Wahrscheinlichkeiten, die das Modell ausgibt)

In [12]:
import torch

texts = [
    "Sehr geehrter Herr Präsident! Liebe Kolleginnen und Kollegen! Das Coronavirus hat uns mit voller Wucht getroffen. Das Ausmaß der Pandemie – wir erleben das auch heute angesichts neuer Infektionszahlen – ist historisch. Es ist allerdings eine Krise, die die gesamte Welt erfasst hat, und in vielen Ländern ist aus dieser Gesundheitskrise längst mehr geworden, nämlich eine dramatische soziale Krise mit Massenarbeitslosigkeit, mit Hunger, mit Obdachlosigkeit, und das auch in entwickelten Industriestaaten. In den USA beispielsweise lag die Arbeitslosigkeit in diesem Jahr streckenweise bei über 14 Prozent. Vor den Food Banks, den amerikanischen Suppenküchen, bildeten sich teilweise kilometerlange Autoschlangen – auf der Suche nach Lebensmitteln. Übrigens, auch das ist ein Grund, warum es gut ist, dass – hoffentlich – jetzt ein amerikanischer Präsident ins Weiße Haus einzieht, der ein soziales Gespür hat.",
    "(Beifall bei der SPD und der CDU/CSU sowie des Abg. Johannes Vogel [Olpe] [FDP])",
    "Meine Damen und Herren, auch bei uns hat die Pandemie unsere Wirtschaft schwer erschüttert. Trotz dieser massiven wirtschaftlichen Erschütterung ist in Deutschland aus dieser Pandemie kein soziales Erdbeben geworden.",
    "(Zuruf von der AfD]: Noch nicht!)",
    "Auch bei uns ist die Arbeitslosenquote durch die Krise um 1,1 Prozentpunkte gegenüber dem Vorjahr auf 5,9 Prozent gestiegen. Aber ich will sagen: Angesichts der massiven wirtschaftlichen Erschütterungen ist das weit weniger, als es zu befürchten stand. Liegt das daran, dass wir mehr Glück gehabt haben als die Amerikaner?",
    "Die Antwort ist: Nein, das liegt daran, dass wir in Deutschland einen der leistungsfähigsten Sozialstaaten der Erde haben,",
    "(Beifall bei der SPD und der CDU/CSU sowie bei Abgeordneten der FDP)",
    "einen Sozialstaat, der sich gerade in dieser Krise auch bewährt, einen Sozialstaat, in dem übrigens seit Monaten und auch im Moment Zehntausende von Beschäftigten für soziale Sicherheit sorgen.",
    "Ob die Pflegerin in der Klinik oder der Sachbearbeiter in der Arbeitsagentur: Dieser Sozialstaat ist einer, indem viele Menschen jetzt mithelfen, dass Hilfen auch ankommen. Dieser Sozialstaat, meine Damen und Herren – ich sage das gerade in Bezug auf die Vorrede – wirkt auch wie ein Immunsystem für unsere Gesellschaft. Dieses Immunsystem schützt uns vor den sozialen, aber eben auch den politischen Infektionen in der Krise; denn da, wo Vertrauen in soziale Sicherheit verloren geht, gewinnen politische Scharlatane. Lassen Sie mich das deutlich sagen: Egal ob sie innerlich Pickelhaube, Stahlhelm oder Aluhut tragen, das werden wir in Deutschland nicht zulassen.",
    "(Beifall bei der SPD, der CDU/CSU, der FDP und dem GRÜNE sowie des Abg. Dr. Dietmar Bartsch [DIE LINKE])",
    "Herr Präsident, meine Damen und Herren, wir debattieren in dieser Woche einen Haushalt mit außergewöhnlich hohen Ausgaben. Dazu gehören auch die Ausgaben für das Kurzarbeitergeld, für den erleichterten Zugang zur Grundsicherung und die Ausbildungsprämie. Wir haben mit dem Kurzarbeitergeld dafür gesorgt, dass in der Krise Millionen von Arbeitsplätzen in Deutschland gesichert wurden. Es war richtig und vernünftig, dass wir diese Regeln, auch mit Blick auf die aktuelle Situation, in das nächste Jahr verlängert haben. In der Autobranche waren zwischenzeitlich fast 60 Prozent der Beschäftigten in Kurzarbeit, im Gastgewerbe sogar noch viel, viel mehr. In einzelnen Regionen war es zum Teil so, dass jeder zweite Beschäftigte in Kurzarbeit war, beispielsweise in Weiden in der Oberpfalz. Keiner von uns will sich vorstellen, wie es in diesen Branchen und Regionen ausgesehen hätte, wenn es kein Kurzarbeitergeld gegeben hätte. Frau Schielke-Ziesing, das, was Sie sagen, ist wahrheitswidrig. Die meisten Menschen sind von Kurzarbeit in Beschäftigung zurückgekehrt.",
    "(Beifall bei der SPD und der CDU/CSU)",
    "Wir hatten im Mai 6 Millionen Menschen in Kurzarbeit, jetzt sind es 2,5 Millionen; die Zahl steigt wieder an. Aber Kurzarbeit ist eine Brücke in Arbeit und in der Regel eben nicht in Arbeitslosigkeit.",
    "(Beifall bei der SPD, der CDU/CSU und der FDP sowie bei Abgeordneten der LINKEN)",
    "Ja, es ist richtig: Kurzarbeit kostet sehr viel Geld. Seit Jahresbeginn haben wir fast 20 Milliarden Euro für das konjunkturelle Kurzarbeitergeld und die Erstattung von Sozialversicherungsbeiträgen für die Betriebe aufgewandt. Aber ich sage Ihnen auch: Massenarbeitslosigkeit und die Rückkehr von Massenarbeitslosigkeit wäre für dieses Land finanziell, wirtschaftlich und sozial viel teurer.",
    "(Beifall bei der SPD sowie bei Abgeordneten der CDU/CSU)",
    "Wir haben die Kraft, dafür zu sorgen, dass in dieser Krise dauerhafte Massenarbeitslosigkeit nach Deutschland nicht zurückkehrt. Da hilft die Kurzarbeit. Deshalb dürfen Sie dieses Instrument nicht so madig machen, wie Sie das hier gerade getan haben.",
    "(Beifall bei der SPD und der CDU/CSU sowie bei Abgeordneten der LINKEN – Widerspruch bei der AfD)",
    "Arbeit, meine Damen und Herren, hat einen Wert für diese Gesellschaft, einen Wert auch für gesellschaftlichen Zusammenhalt, einen Wert aber auch für die Menschen an sich, nämlich selbstbestimmt leben zu können. Es gibt aber auch eine andere Seite, wenn wir über den Wert der Arbeit reden. Wir haben in dieser Coronakrise erlebt, dass viele Menschen, die den Laden am Laufen halten, viel Applaus bekommen haben, aber zu wenig Lohn. Deshalb war es wichtig, dass wir gerade in diesen Zeiten, Frau Schielke-Ziesing, die Grundrente nicht gestrichen haben, sondern sie für die Menschen, die ihr Lebtag gearbeitet haben – das sind vor allem Frauen – und aufgrund von viel zu niedrigen Löhnen nicht genug Rente bekommen, einzuführen. Sie wird zum 1. Januar allen Widerständen zum Trotz in diesem Land eingeführt, meine Damen und Herren.",
    "(Beifall bei der SPD sowie bei Abgeordneten der CDU/CSU)",
    "Es ist auch richtig, dass der Mindestlohn steigt. Die Mindestlohnkommission hat entsprechende Empfehlungen auf den Weg gebracht.",
    "– Ja, kleinen Moment mal.",
    "Die Kommission wird zum 1. Januar einen Schritt in die richtige Richtung gehen.",
    "Wir feiern jetzt fünf Jahre Mindestlohn.",
    "Der Mindestlohn hat übrigens – im Gegensatz zu dem, was vor fünf Jahren behauptet wurde – keine Arbeitsplätze vernichtet. Im Gegenteil: Er hat die Kaufkraft gestärkt.",
    "(Beifall bei Abgeordneten der SPD)",
    "Ich sage aber auch: Die Mindestlohnerhöhung kann für die Menschen, die jetzt den Laden am Laufen halten, die zum Beispiel Regale in unseren Supermärkten einräumen, nur ein Zwischenschritt sein. Wir müssen – darüber werden wir reden; ich werde dazu Vorschläge machen – den Mindestlohn weiter erhöhen und weiterentwickeln, meine Damen und Herren. Auch das ist notwendig.",
    "(Beifall bei der SPD)",
    "Aber vor allen Dingen müssen wir dafür sorgen, dass wir oberhalb des Mindestlohns wieder zu mehr Tarifbindung in diesem Land kommen; denn Tarifbindung heißt in der Regel bessere Löhne und Gehälter.",
    "Wir haben über viele Jahrzehnte einen Rückgang der Tarifbindung in Deutschland erlebt, zum Beispiel im Einzelhandel, wo nur noch 20 Prozent der Beschäftigten tarifgebunden sind, aber zum Beispiel auch in der Altenpflege, wo auch nur 20 Prozent der Beschäftigten tarifgebunden sind. Wir reden alle immer so schön über die Helden und Heldinnen des Alltags, und wir klatschen hier im Bundestag auch ganz kräftig für sie.",
    "Aber Klatschen zahlt für die Pflegerinnen und Pfleger keine Rechnung und keine Miete. Deshalb haben wir nicht nur den Mindestlohn erhöht, und zwar differenziert erhöht, auch für Pflegehilfskräfte und für qualifizierte Pflegekräfte, sondern wir haben die Chance für einen allgemeinverbindlichen Tarifvertrag eröffnet. Wenn der Antrag vorliegen wird, wir ihn prüfen und die Voraussetzungen erfüllt sind, will ich, dass wir im nächsten Jahr für die Altenpflege diesen Tarifvertrag für allgemeinverbindlich erklären, meine Damen und Herren.",
    "(Beifall bei der SPD sowie bei Abgeordneten der CDU/CSU und des GRÜNE)",
    "Es geht also um den Wert der Arbeit in der Krise. Aber es geht auch um die Würde der Arbeit. Der Harvard-Philosoph Michael Sandel hat es auf den Punkt gebracht – ich zitiere mit der Genehmigung des Präsidenten –: Unser größtes Problem ist die wachsende Ungleichheit, nicht nur mit Blick auf Einkommen und Wohlstand, sondern auch mit Blick auf die soziale Wertschätzung. – Ich sage das, weil wir in der Krise auch erlebt haben, dass es an Wertschätzung gefehlt hat, zum Beispiel in der Fleischindustrie, wo wir seit vielen Jahren ein System der Ausbeutung erlebt haben, das unter den Bedingungen der Pandemie zu einem großen Gesundheitsrisiko für viele Menschen geworden ist. Deshalb bin ich den Koalitionsfraktionen außerordentlich dankbar, dass wir in der nächsten Woche – da bin ich zuversichtlich – das Arbeitsschutzkontrollgesetz durchsetzen, um mit solchen ausbeuterischen Verhältnissen aufzuräumen.",
    "(Beifall bei der SPD sowie bei Abgeordneten der CDU/CSU)",
    "Ich habe aber bei den Koalitionsfraktionen auch die Hoffnung, dass wir, wenn wir über die Würde der Arbeit reden, nicht nur den nationalen Blick haben, sondern im Kampf um menschenwürdige Arbeitsbedingungen unserer Verantwortung auch weltweit gerecht werden. Deshalb setze ich darauf, dass wir zu einer Einigung beim Lieferkettengesetz kommen. Meine Damen und Herren, es geht an dieser Stelle nicht um Kapitalinteressen, sondern es geht um Menschenwürde.",
    "(Beifall bei der SPD sowie den Abg. Hermann Gröhe [CDU/CSU] und Matthias W. Birkwald [DIE LINKE])",
    "Es geht also um den Wert der Arbeit und die Würde der Arbeit. Es geht aber gerade in dieser Pandemie auch um den Wandel der Arbeit; denn Corona wird vieles beschleunigen: in der Digitalisierung, im Strukturwandel unserer Industrie, aber auch in der Art und Weise, wie wir arbeiten. Deshalb ist es wichtig, dass wir in der Krise nicht nur Krisenmanagement machen, sondern auch Zukunftsvorsorge betreiben. Das tun wir in diesem Haushalt, indem wir beispielsweise mithelfen, den Instrumenten, die wir jetzt geschaffen haben, Flügel zu verleihen, damit die Beschäftigten von heute auch die Chance haben, die Arbeit von morgen zu machen. Es geht um die Instrumente, die die Kurzarbeit mit Qualifizierung verbinden, die Instrumente, die vor allen Dingen kleinen und mittelständischen Unternehmen bei der Fachkräftesicherung helfen. Im Sinne der Beschäftigungsfähigkeit der Arbeitnehmerinnen und Arbeitnehmer sorgen wir damit dafür, die Krise für Qualifizierung und Weiterbildung zu nutzen. Es geht aber auch um das mobile Arbeiten. Wir arbeiten an einem Rechtsrahmen, der mithilft, dieser neuen Realität für viele Beschäftigte Rechnung zu tragen, indem wir da, wo es möglich ist, den Beschäftigten rechtlich den Rücken stärken, auch einmal ein, zwei Tage Homeoffice machen zu können, aber auch dafür sorgen, dass Homeoffice nicht zur Entgrenzung von Arbeit und Privatleben führt. Auch im Homeoffice, meine Damen und Herren, muss einmal Feierabend sein. Das ist Teil des Gesundheitsschutzes der Beschäftigten.",
    "(Beifall bei der SPD sowie bei Abgeordneten der CDU/CSU)",
    "Herr Präsident, meine Damen und Herren, es geht in diesen Zeiten darum, unser Land zusammenzuhalten. Wir haben nach wie vor schwierige Monate vor uns. Am Wochenende sind Entscheidungen zu treffen, die wiederum wahrscheinlich unser gesellschaftliches und wirtschaftliches Leben einschränken werden, um die Gesundheit von Menschen zu schützen. Aber Sie können sich, meine Damen und Herren, auch draußen an den Bildschirmen, darauf verlassen: Diese Bundesregierung kämpft um jeden Arbeitsplatz. Diese Bundesregierung weiß um den Wert, die Würde und den Wandel der Arbeit, und das kennzeichnet auch diesen Haushalt. Herzlichen Dank.",
    "(Beifall bei der SPD und der CDU/CSU)"
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

predictions = np.argmax(logits.numpy(), axis=-1)

# Extrahiert die Indizes der vorhergesagten "Einwurf" Labels
einwurf_indices = [i for i, prediction in enumerate(predictions) if prediction == 1]

# Ergebnis entspricht den Indizes der Sätze der texts Liste. Es wird bei 0 angefangen zu zählen.
print("Predicted Einwurf Indices: ", einwurf_indices)

for text, prediction in zip(texts, predictions):
    label = "Einwurf" if prediction == 1 else "Kein Einwurf"
    print(f"Satz: {text} -> Vorhergesagt: {label}")

Predicted Einwurf Indices:  [3, 5, 6, 11, 13, 15, 17, 19, 25, 27, 31, 33, 37, 39]
Satz: Sehr geehrter Herr Präsident! Liebe Kolleginnen und Kollegen! Das Coronavirus hat uns mit voller Wucht getroffen. Das Ausmaß der Pandemie – wir erleben das auch heute angesichts neuer Infektionszahlen – ist historisch. Es ist allerdings eine Krise, die die gesamte Welt erfasst hat, und in vielen Ländern ist aus dieser Gesundheitskrise längst mehr geworden, nämlich eine dramatische soziale Krise mit Massenarbeitslosigkeit, mit Hunger, mit Obdachlosigkeit, und das auch in entwickelten Industriestaaten. In den USA beispielsweise lag die Arbeitslosigkeit in diesem Jahr streckenweise bei über 14 Prozent. Vor den Food Banks, den amerikanischen Suppenküchen, bildeten sich teilweise kilometerlange Autoschlangen – auf der Suche nach Lebensmitteln. Übrigens, auch das ist ein Grund, warum es gut ist, dass – hoffentlich – jetzt ein amerikanischer Präsident ins Weiße Haus einzieht, der ein soziales Gespür hat. -

In dieser Zelle werden die Evaluationsmetriken, wie Accuracy, Recall, Precision und dessen harmonisches Mittel der F1-Score ausgegeben. 
Mit einem Wert von 0,98 bei beiden Klassen ist die Precision sehr gut. Also 98% der Instanzen, die als "Kein Einwurf" klassifiziert wurde, sind tatsächlich "Kein Einwurf". Genauso verhält das Modell sich bei der Klasse "Einwurf".
Interessant ist, dass der recall bei der Klasse "Kein Einwurf" etwas höher ist als bei "Einwurf". Das bedeutet, dass 99% der Instanzen, die tatsächlich "Kein Einwurf" sind auch korrekt als "Kein Einwurf" erkannt wurden. Bei der Klasse "Einwurf" wurden nur 97% aller Instanzen korrekt erkannt.


Dafür habe ich die prediction_pipeline verwendet, welche einen Text als Eingabe nimmt und die Vorhersagepipeline benutzt, um eine Klassifizierung durchzuführen. In der Vorhersagepipeline werden das zuvor gespeicherte Modell und der Tokenizer geladen. 

In [13]:
from tqdm import tqdm

prediction_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer)

def predict_interruption(text):
    #max_lengtgh wichtig, weil die maximale Tokenlänge von BERT 512 beträgt
    result = prediction_pipeline(text, truncation=True, max_length=512)[0] 
    return int(result['label'].split('_')[-1])

predict_df = df.copy()
tqdm.pandas()
predict_df['predicted_label'] = predict_df['text'].progress_apply(predict_interruption)

accuracy = (predict_df['label'] == predict_df['predicted_label']).mean()
print(f"BERT Model Accuracy: {accuracy:.2f}")

from sklearn.metrics import confusion_matrix, classification_report

conf_matrix = confusion_matrix(predict_df['label'], predict_df['predicted_label'])
print(conf_matrix)

print(classification_report(predict_df['label'], predict_df['predicted_label'], target_names=['Kein Einwurf', 'Einwurf']))

100%|██████████| 5364/5364 [11:29<00:00,  7.77it/s]

BERT Model Accuracy: 0.98
[[3142   40]
 [  63 2119]]
              precision    recall  f1-score   support

Kein Einwurf       0.98      0.99      0.98      3182
     Einwurf       0.98      0.97      0.98      2182

    accuracy                           0.98      5364
   macro avg       0.98      0.98      0.98      5364
weighted avg       0.98      0.98      0.98      5364

